# 🔗 Multi-Hop Research QA

## What We're Building

A QA system that answers questions requiring **evidence from multiple documents**
by decomposing complex questions into sub-questions and gathering evidence step-by-step.

## The Problem

```
Simple question:  "What is Python?"           → 1 retrieval step
Multi-hop question: "Is the language that Django
  is written in older than JavaScript?"        → Need to find:
                                                  1. Django → written in Python
                                                  2. Python → created 1991
                                                  3. JavaScript → created 1995
                                                  4. Compare: 1991 < 1995 → Yes
```

## Architecture

```
User Question
     │
     ▼
┌─────────────┐
│  Decompose   │  → Break into sub-questions
└──────┬──────┘
       ▼
┌─────────────┐
│  Research    │  → Retrieve + answer each sub-question
│  Loop       │  → Feed prior answers as context
└──────┬──────┘
       ▼
┌─────────────┐
│  Synthesize  │  → Combine all evidence into final answer
└─────────────┘
```

## Stack
- **LangGraph** — orchestrate the multi-step research loop
- **ChromaDB** — vector store
- **Ollama** — local LLM

## Step 1 — Imports & Multi-Topic Knowledge Base

In [ ]:
from typing import TypedDict
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from langgraph.graph import StateGraph, END

print("All imports successful!")

In [ ]:
# Multi-topic knowledge base — answering cross-domain questions
# requires evidence from multiple chunks across topics
documents = [
    # Programming Languages
    Document(page_content="Python was created by Guido van Rossum and first released in 1991. It uses dynamic typing and garbage collection. Python is the most popular language for data science and machine learning.", metadata={"topic": "languages"}),
    Document(page_content="JavaScript was created by Brendan Eich at Netscape in 1995. It took only 10 days to build the first version. JavaScript is the language of the web, running in all browsers natively.", metadata={"topic": "languages"}),
    Document(page_content="Rust was first released in 2015 by Mozilla. It offers memory safety without garbage collection through its ownership system. Rust is used in Firefox, Dropbox, and Cloudflare.", metadata={"topic": "languages"}),
    Document(page_content="Go (Golang) was created at Google by Robert Griesemer, Rob Pike, and Ken Thompson. It was released in 2009. Go is designed for concurrent programming and compiles to native code.", metadata={"topic": "languages"}),

    # Frameworks
    Document(page_content="Django is a Python web framework created in 2005. It follows the MVT (Model-View-Template) pattern and includes an ORM, admin panel, and authentication out of the box.", metadata={"topic": "frameworks"}),
    Document(page_content="React is a JavaScript library for building UIs, created by Facebook in 2013. It uses a virtual DOM and component-based architecture. React has over 200k GitHub stars.", metadata={"topic": "frameworks"}),
    Document(page_content="FastAPI is a modern Python web framework released in 2018 by Sebastian Ramirez. It's built on Starlette and Pydantic, offering automatic OpenAPI docs and async support.", metadata={"topic": "frameworks"}),
    Document(page_content="Next.js is a React framework by Vercel for server-side rendering, static generation, and API routes. It was first released in 2016 and is widely used for production React apps.", metadata={"topic": "frameworks"}),

    # Companies
    Document(page_content="Google was founded by Larry Page and Sergey Brin in 1998. Key products: Search, Gmail, YouTube, Android, GCP. Revenue in 2023: $307 billion. Google created Go, Angular, and TensorFlow.", metadata={"topic": "companies"}),
    Document(page_content="Meta (formerly Facebook) was founded by Mark Zuckerberg in 2004. Key products: Facebook, Instagram, WhatsApp, Oculus VR. Meta created React, PyTorch, and GraphQL.", metadata={"topic": "companies"}),
    Document(page_content="Microsoft was founded by Bill Gates and Paul Allen in 1975. Key products: Windows, Office, Azure, GitHub, VS Code. Microsoft acquired GitHub in 2018 for $7.5 billion.", metadata={"topic": "companies"}),
    Document(page_content="Mozilla is a non-profit organization that developed Firefox browser. Mozilla created Rust programming language. Mozilla's mission is to keep the internet open and accessible.", metadata={"topic": "companies"}),

    # ML Frameworks
    Document(page_content="TensorFlow was developed by Google Brain and released in 2015. It supports distributed training, mobile deployment (TFLite), and production serving (TF Serving).", metadata={"topic": "ml"}),
    Document(page_content="PyTorch was developed by Meta's AI Research lab and released in 2016. It uses dynamic computation graphs (eager mode) and is the preferred framework for research papers.", metadata={"topic": "ml"}),
    Document(page_content="scikit-learn is a Python library for classical ML algorithms. It was first released in 2007. It provides implementations of SVM, random forest, clustering, and preprocessing tools.", metadata={"topic": "ml"}),
]

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(documents, embeddings, collection_name="multihop")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Indexed {len(documents)} documents across {len(set(d.metadata['topic'] for d in documents))} topics")

## Step 2 — Question Decomposer

The LLM breaks a complex question into simpler sub-questions.

In [ ]:
decompose_prompt = ChatPromptTemplate.from_template(
    """Break this complex question into 2-4 simpler sub-questions that,
when answered individually, provide all the information needed to answer
the original question.

Complex question: {question}

Output each sub-question on a new line, numbered 1., 2., etc.
Sub-questions:"""
)


def decompose_question(question: str) -> list[str]:
    chain = decompose_prompt | llm | StrOutputParser()
    result = chain.invoke({"question": question})
    # Parse numbered lines
    sub_qs = []
    for line in result.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            # Remove numbering
            parts = line.split(".", 1)
            if len(parts) > 1:
                sub_qs.append(parts[1].strip())
    return sub_qs if sub_qs else [question]  # fallback to original


# Test decomposition
q = "Is the language that Django is written in older than the language that React is written in?"
sub_qs = decompose_question(q)
print(f"Original: {q}\n")
for i, sq in enumerate(sub_qs, 1):
    print(f"  Sub-Q {i}: {sq}")

## Step 3 — Build LangGraph Multi-Hop Pipeline

The graph processes sub-questions one at a time, accumulating evidence.

In [ ]:
class ResearchState(TypedDict):
    """State flowing through the multi-hop research graph."""
    original_question: str
    sub_questions: list[str]
    current_index: int
    evidence: list[dict]  # [{"question": ..., "answer": ..., "sources": ...}]
    final_answer: str


# --- Node 1: Decompose ---
def decompose_node(state: ResearchState) -> dict:
    sub_qs = decompose_question(state["original_question"])
    print(f"  📋 Decomposed into {len(sub_qs)} sub-questions")
    for i, q in enumerate(sub_qs, 1):
        print(f"     {i}. {q}")
    return {"sub_questions": sub_qs, "current_index": 0, "evidence": []}


# --- Node 2: Research one sub-question ---
research_prompt = ChatPromptTemplate.from_template(
    """Answer this sub-question using the provided evidence.
If you can't answer from the evidence, say "I don't have enough information."

Previous findings (from earlier sub-questions):
{prior_evidence}

Retrieved documents:
{retrieved}

Sub-question: {sub_question}

Answer (be specific, cite facts):"""
)


def research_node(state: ResearchState) -> dict:
    idx = state["current_index"]
    sub_q = state["sub_questions"][idx]

    # Retrieve documents for this sub-question
    docs = retriever.invoke(sub_q)
    retrieved = "\n\n".join(d.page_content for d in docs)

    # Format prior evidence
    prior = "None yet." if not state["evidence"] else "\n".join(
        f"- Q: {e['question']} → A: {e['answer']}" for e in state["evidence"]
    )

    # Answer
    chain = research_prompt | llm | StrOutputParser()
    answer = chain.invoke({"sub_question": sub_q, "retrieved": retrieved, "prior_evidence": prior})

    print(f"  🔍 Sub-Q {idx+1}: {sub_q}")
    print(f"     → {answer[:100]}..." if len(answer) > 100 else f"     → {answer}")

    new_evidence = state["evidence"] + [{
        "question": sub_q,
        "answer": answer,
        "sources": [d.page_content[:50] for d in docs]
    }]
    return {"evidence": new_evidence, "current_index": idx + 1}


# --- Node 3: Synthesize final answer ---
synthesize_prompt = ChatPromptTemplate.from_template(
    """You have researched the following sub-questions. Synthesize the
findings into a clear, comprehensive answer to the original question.

Original question: {original_question}

Research findings:
{evidence}

Final answer:"""
)


def synthesize_node(state: ResearchState) -> dict:
    evidence_text = "\n\n".join(
        f"Sub-question: {e['question']}\nAnswer: {e['answer']}" for e in state["evidence"]
    )
    chain = synthesize_prompt | llm | StrOutputParser()
    answer = chain.invoke({"original_question": state["original_question"], "evidence": evidence_text})
    print(f"  ✅ Synthesized final answer!")
    return {"final_answer": answer}


# --- Routing ---
def route_research(state: ResearchState) -> str:
    if state["current_index"] < len(state["sub_questions"]):
        return "research"  # more sub-questions to answer
    return "synthesize"  # all done, combine


# --- Build the graph ---
workflow = StateGraph(ResearchState)
workflow.add_node("decompose", decompose_node)
workflow.add_node("research", research_node)
workflow.add_node("synthesize", synthesize_node)

workflow.set_entry_point("decompose")
workflow.add_conditional_edges("decompose", route_research)
workflow.add_conditional_edges("research", route_research)
workflow.add_edge("synthesize", END)

graph = workflow.compile()
print("Multi-hop research graph built!")

## Step 4 — Run Multi-Hop Queries

In [ ]:
def multi_hop_qa(question: str) -> str:
    print(f"\n{'='*60}")
    print(f"❓ {question}")
    print("=" * 60)
    result = graph.invoke({
        "original_question": question,
        "sub_questions": [],
        "current_index": 0,
        "evidence": [],
        "final_answer": ""
    })
    print(f"\n📝 Final Answer:")
    print(result["final_answer"])
    return result["final_answer"]


# Multi-hop: needs Django → Python → 1991 and React → JavaScript → 1995
_ = multi_hop_qa("Is the language that Django is written in older than the language that React is written in?")

In [ ]:
# Multi-hop: needs to find ML frameworks + which companies created them
_ = multi_hop_qa("Which company created both a programming language and a deep learning framework?")

In [ ]:
# Multi-hop: compare across categories
_ = multi_hop_qa("What is the newest programming language in the knowledge base, and does it have a web framework?")

## Step 5 — Compare Single-Hop vs Multi-Hop

In [ ]:
single_hop_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the provided context.

Context:
{context}

Question: {question}

Answer:"""
)


def single_hop_qa(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    chain = single_hop_prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": question})


# Compare on a multi-hop question
question = "Did the company that created React also create a deep learning framework? If so, which one?"

print("📌 Single-hop (one retrieval):")
print(single_hop_qa(question))

print("\n" + "-"*40)

print("\n📌 Multi-hop (decomposed):")
_ = multi_hop_qa(question)

## 🧠 Key Concepts Recap

| Concept | What Happens |
|---------|-------------|
| **Decomposition** | Complex question → multiple simple sub-questions |
| **Sequential Research** | Each sub-question retrieves + gets answered |
| **Evidence Accumulation** | Later sub-questions see earlier answers as context |
| **Synthesis** | All findings combined into a comprehensive final answer |

## When to Use Multi-Hop

- Comparison questions: "Is X bigger/older/better than Y?"
- Bridge questions: "What did the founder of X create later?"
- Aggregation: "How many companies created open-source ML tools?"

## 🔧 Extensions

- **Adaptive depth**: Let the LLM decide if more hops are needed
- **Parallel sub-questions**: Research independent sub-questions concurrently
- **Evidence scoring**: Weight evidence by retrieval confidence